# Cliente Analítico — Exemplo (Grupo 2)

Este notebook é um **exemplo** de como o Grupo 2 deve consumir o microsserviço de ML que vocês mesmos construíram. Ele NÃO importa as funções Python diretamente — usa apenas requisições HTTP, exatamente como uma aplicação web em produção faria.

## Cenário deste exemplo

- **Bloco escolhido:** D (Busca Semântica)
- **Sistema-alvo:** IBGE — Localidades (lista de municípios brasileiros)
- **Objetivo:** demonstrar que é possível buscar municípios por descrição livre em PT-BR ("cidades industriais do Sudeste") em vez de ter que saber o nome exato.

Adapte para o sistema-alvo da sua equipe.

## 1. Pré-requisitos

Antes de rodar este notebook, garanta que:

1. O microsserviço foi treinado: `python train.py --bloco D --datasource api`
2. O microsserviço está no ar: `uvicorn app.main:app --port 8000`
3. As dependências estão instaladas: `pip install requests pandas matplotlib`

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

API = "http://localhost:8000"

## 2. Healthcheck — o serviço está vivo?

Sempre comece por aqui. Se o `/health` falhar, todo o resto vai falhar também.

In [ ]:
resp = requests.get(f"{API}/health", timeout=5)
resp.raise_for_status()
resp.json()

## 3. Consultas-de-prova

Aqui demonstramos o valor agregado: queries em linguagem natural devolvem municípios geograficamente coerentes mesmo sem palavras em comum.

**Importante:** essas queries são qualitativas, para inspeção visual. A avaliação quantitativa (Recall@k, MRR) deve ser feita em uma amostra rotulada manualmente — veja a Seção 5 deste notebook.

In [ ]:
queries_demo = [
    "capital do nordeste",
    "cidade litorânea de Pernambuco",
    "município no interior de São Paulo",
    "cidade da região metropolitana do Rio",
]

for q in queries_demo:
    r = requests.get(f"{API}/search", params={"q": q, "k": 5}, timeout=10)
    r.raise_for_status()
    hits = r.json()["hits"]
    print(f"\n>> Query: {q!r}")
    for h in hits:
        print(f"   {h['titulo']:30s}  score={h['score']:.3f}")

## 4. Latência

Em produção, a busca semântica precisa responder em menos de 200ms para não travar a experiência do usuário. Vamos medir.

In [ ]:
import time

tempos = []
for _ in range(50):
    t0 = time.perf_counter()
    requests.get(f"{API}/search", params={"q": "capital nordeste", "k": 10}, timeout=10).json()
    tempos.append((time.perf_counter() - t0) * 1000)

df_t = pd.DataFrame({"latencia_ms": tempos})
print(df_t.describe())
df_t.plot.hist(bins=20, title="Latência por requisição (ms)")
plt.xlabel("ms")
plt.show()

## 5. Avaliação quantitativa (placeholder)

**Esta é a parte que vocês precisam preencher.** Crie uma planilha `data/queries_rotuladas.csv` com pelo menos 30 queries e, para cada uma, a lista de IDs de municípios considerados "relevantes" (julgamento humano). Calcule:

- **Recall@k** — fração de relevantes que apareceram nos top-k.
- **MRR (Mean Reciprocal Rank)** — média de 1/posição-do-primeiro-relevante.

Estas métricas serão a base do relatório técnico (Seção 4 do entregável).

## 6. Estudo de viabilidade — questões para responder

Use estas perguntas para guiar o entregável "Estudo de Viabilidade" (15% da nota):

1. **Quem se beneficiaria** se este componente fosse plugado ao sistema do IBGE?
2. Que **ganho concreto** ele entrega vs. a busca textual atual?
3. Quais **riscos éticos**? (ex: viés contra municípios pequenos com pouca descrição)
4. Custo de **manutenção**: o índice precisa ser reconstruído quando? Quem opera?
5. **LGPD**: há dados pessoais aqui? (Resposta para este caso: não.)